In [1]:
from ultralytics import YOLO
import cv2 as cv
import torch

## Custom Trained Model

In [2]:
# Import model
model = YOLO('DisposableSingle12.pt')

# Export the model to ONNX format
# model.export(format="onnx")

Ultralytics 8.3.21  Python-3.12.1 torch-2.4.1+cu124 CPU (AMD Ryzen 7 7800X3D 8-Core Processor)
YOLO11l summary (fused): 464 layers, 25,282,396 parameters, 0 gradients, 86.6 GFLOPs

PyTorch: starting from 'DisposableSingle12.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (48.9 MB)

ONNX: starting export with onnx 1.16.1 opset 19...
ONNX: slimming with onnxslim 0.1.39...
ONNX: export success  2.8s, saved as 'DisposableSingle12.onnx' (96.8 MB)

Export complete (3.7s)
Results saved to C:\Users\Trever\OneDrive\Python\CV\YOLO Testing
Predict:         yolo predict task=detect model=DisposableSingle12.onnx imgsz=640  
Validate:        yolo val task=detect model=DisposableSingle12.onnx imgsz=640 data=C:/Users/Trever/Desktop/Dataset/Final/Versions/v8/data.yaml  
Visualize:       https://netron.app


'DisposableSingle12.onnx'

In [3]:
# YOLO tracking method
results = model.track(source=0, show=True, conf=0.6, tracker='botsort.yaml', save=False)

In [4]:
# Custom live tracking logic

# Initialize the video capture
cap = cv.VideoCapture(0)

# Main loop for video capture and detection
while True:
    # Capture each frame from the webcam
    ret, image = cap.read()

    if not ret:
        print("Failed to grab frame")
        break

    # Run inference with the model
    results = model(image, conf=0.6)

    # Process detection results
    for info in results:
        for box in info.boxes:
            # Move tensors to CPU and convert to numpy for processing
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            confidence = int(box.conf[0].cpu().numpy() * 100)
            class_num = int(box.cls[0].cpu())
            class_name = results[0].names[class_num]

            # Draw bounding box, class name, and confidence
            cv.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 3)
            cv.putText(image, f"{class_name} {confidence}%", (x1, y1 - 10),
                       cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Resize the output image to make the window larger
    resized_image = cv.resize(image, (960, 720))  # Resize to desired dimensions

    # Display the frame
    cv.imshow('Object Detection', resized_image)

    # Break on pressing 'q' key
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv.destroyAllWindows()